In [68]:
#loading packages and datasets
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [69]:
#problem 2
#removing useless columns
drop_cols = ["Unnamed: 0", "id", "date", "zipcode"]

train = train.drop(columns=drop_cols, errors = "ignore")
test = test.drop(columns=drop_cols, errors = "ignore")

train.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,lat,long,sqft_living15,sqft_lot15
0,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,47.5112,-122.257,1340,5650
1,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,47.7210,-122.319,1690,7639
2,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,47.7379,-122.233,2720,8062
3,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,47.5208,-122.393,1360,5000
4,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,47.6168,-122.045,1800,7503


In [70]:
#scaling the data
train["price"] = train["price"] / 1000
test["price"] = test["price"] / 1000

In [71]:
#splitting the features/target
X_train = train.drop(columns=["price"])
y_train = train["price"]

X_test = test.drop(columns=["price"])
y_test = test["price"]

In [72]:
#standardizing train features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [73]:
#training the linear regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

LinearRegression()

In [74]:
#coefficients
coefficients = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model.coef_
}).sort_values(by="coefficient", key=abs, ascending=False)

print(coefficients)
print("Intercept:", model.intercept_)

          feature  coefficient
8           grade    92.231475
13            lat    78.375737
11       yr_built   -67.643117
5      waterfront    63.742900
2     sqft_living    56.748837
9      sqft_above    48.290089
6            view    48.200109
15  sqft_living15    45.577658
10  sqft_basement    27.137032
1       bathrooms    18.527633
12   yr_renovated    17.271380
7       condition    12.964269
16     sqft_lot15   -12.930091
0        bedrooms   -12.521962
3        sqft_lot    10.881868
4          floors     8.043721
14           long    -1.035203
Intercept: 520.414834000001


In [75]:
#Q1. training metrics
train_pred = model.predict(X_train_scaled)

train_mse = mean_squared_error(y_train, train_pred)
train_r2 = r2_score(y_train, train_pred)

print("Train MSE:", train_mse)
print("Train R2:", train_r2)

Train MSE: 31486.16777579488
Train R2: 0.7265334318706018


In [76]:
#Q2. testing metrics
test_pred = model.predict(X_test_scaled)

test_mse = mean_squared_error(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)

print("Test MSE:", test_mse)
print("Test R2:", test_r2)

Test MSE: 57628.154705670386
Test R2: 0.6543560876120954


In [77]:
#problem 3, part 1
#closed form needs the intercept term
def add_bias(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack([ones, X])

Xtr = add_bias(X_train_scaled)
Xte = add_bias(X_test_scaled)

#closed form functions
def closed_form_fit(X, y):
    return np.linalg.pinv(X.T @ X) @ X.T @ y
def closed_form_predict(X, theta):
    return X @ theta

theta = closed_form_fit(Xtr, y_train.values)

train_pred_cf = closed_form_predict(Xtr, theta)
test_pred_cf  = closed_form_predict(Xte, theta)

In [78]:
#part 2
cf_train_mse = mean_squared_error(y_train, train_pred_cf)
cf_test_mse  = mean_squared_error(y_test, test_pred_cf)
cf_train_r2 = r2_score(y_train, train_pred_cf)
cf_test_r2  = r2_score(y_test, test_pred_cf)

print("Closed Form Train MSE:", cf_train_mse)
print("Closed Form Test MSE:", cf_test_mse)
print("Closed Form Train R2:", cf_train_r2)
print("Closed Form Test R2:", cf_test_r2)

Closed Form Train MSE: 31486.167775794886
Closed Form Test MSE: 57628.15470567011
Closed Form Train R2: 0.7265334318706018
Closed Form Test R2: 0.6543560876120971


In [79]:
#Problem 4

#get single feature from train/test datasets
x_train = train["sqft_living"].values.astype(float)
x_test  = test["sqft_living"].values.astype(float)

#polynomial design matrix w/ intercept
def polynomials(x, p):
    poly_feats = np.column_stack([x**k for k in range(1, p+1)]) 
    return np.column_stack([np.ones(len(x)), poly_feats])   

#standardize sqft_living w/ train dataset
mu = x_train.mean()
sigma = x_train.std(ddof=0)
x_train_std = (x_train - mu) / sigma
x_test_std  = (x_test - mu) / sigma
  
#train for p <= 5
rows = []
for p in range(1,6):
    Xtr_p = polynomials(x_train_std, p)
    Xte_p = polynomials(x_test_std, p)
    theta_p = closed_form_fit(Xtr_p, y_train.values)
    pred_tr = closed_form_predict(Xtr_p, theta_p)
    pred_te = closed_form_predict(Xte_p, theta_p)
    rows.append({
        "p": p,
        "train_MSE": mean_squared_error(y_train, pred_tr),
        "train_R2":  r2_score(y_train, pred_tr),
        "test_MSE":  mean_squared_error(y_test, pred_te),
        "test_R2":   r2_score(y_test, pred_te),
    })

results_poly = pd.DataFrame(rows)
print(results_poly)

   p     train_MSE  train_R2       test_MSE   test_R2
0  1  57947.526161  0.496709   88575.978543  0.468736
1  2  54822.665116  0.523849   71791.679479  0.569406
2  3  53785.194716  0.532860   99833.483763  0.401216
3  4  52795.774758  0.541453  250979.274285 -0.505331
4  5  52626.111955  0.542927  570616.914820 -2.422464


In [93]:
#problem 5
def gd(X, y, alpha, iters):
    n, d = X.shape
    theta = np.zeros(d)
    
    for i in range(iters):
        preds = X @ theta
        grad = (2/n) * (X.T @ (preds - y))
        theta = theta - alpha * grad
        
    return theta

def eval_theta(Xtr, Xte, ytr, yte, theta):
    pred_tr = Xtr @ theta
    pred_te = Xte @ theta
    
    return {
        "train_MSE": mean_squared_error(ytr, pred_tr),
        "train_R2":  r2_score(ytr, pred_tr),
        "test_MSE":  mean_squared_error(yte, pred_te),
        "test_R2":   r2_score(yte, pred_te),
    }

In [100]:
alphas = [0.01, 0.1, 0.5]
i_list = [10, 50, 100]
rows = []

for a in alphas:
    for it in i_list:
        theta_gd = gd(Xtr, y_train.values, a, it)
        metrics = eval_theta(Xtr, Xte, y_train.values, y_test.values, theta_gd)
        rows.append({
            "alpha": a,
            "iters": it,
            "theta_norm": np.linalg.norm(theta_gd),
            **metrics
        })

gd_results = pd.DataFrame(rows)
print(gd_results)

   alpha  iters    theta_norm      train_MSE       train_R2       test_MSE  \
0   0.01     10  1.221884e+02   2.357278e+05  -1.047365e+00   2.805687e+05   
1   0.01     50  3.633735e+02   6.972050e+04   3.944571e-01   9.704954e+04   
2   0.01    100  4.822708e+02   3.682035e+04   6.802045e-01   6.333304e+04   
3   0.10     10  4.952783e+02   3.510510e+04   6.951019e-01   6.163043e+04   
4   0.10     50  5.525800e+02   3.149726e+04   7.264371e-01   5.772248e+04   
5   0.10    100  5.531809e+02   3.148643e+04   7.265311e-01   5.763896e+04   
6   0.50     10  1.671857e+08   1.456064e+17  -1.264635e+12   1.626068e+17   
7   0.50     50  1.554947e+33   1.259542e+67  -1.093949e+62   1.406601e+67   
8   0.50    100  2.525577e+64  3.322792e+129 -2.885942e+124  3.710745e+129   

         test_R2  
0  -6.828036e-01  
1   4.179133e-01  
2   6.201392e-01  
3   6.303511e-01  
4   6.537904e-01  
5   6.542913e-01  
6  -9.752880e+11  
7  -8.436553e+61  
8 -2.225642e+124  


In [95]:
#problem 6, part 2
def ridge_gd(X, y, alpha, iters, lam, penalize_intercept=False):
    n, d = X.shape
    theta = np.zeros(d)
    for i in range(iters):
        preds = X @ theta
        grad = (2/n) * (X.T @ (preds - y))
        #regularization term
        reg = 2 * lam * theta

        if not penalize_intercept:
            reg[0] = 0
            
        theta = theta - alpha * (grad + reg)
    return theta

In [99]:
#part 3
np.random.seed(123)

N = 1000
X = np.random.uniform(-2, 2, size=N)
e = np.random.normal(0, 2, size=N)   
Y = 1 + 2*X + e

#matrix w/ intercept
Xmat = np.column_stack([np.ones(N), X])

def ridge_cf(X, y, lam, penalize_intercept=False):
    d = X.shape[1]
    I = np.eye(d)
    if not penalize_intercept:
        I[0,0] = 0
    return np.linalg.inv(X.T @ X + lam * I) @ (X.T @ y)

def metrics(X, y, theta):
    preds = X @ theta
    mse = mean_squared_error(y, preds)
    r2 = r2_score(y, preds)
    return mse, r2

lams = [0, 1, 10, 100, 1000, 10000]
rows = []

for lam in lams:
    theta = ridge_cf(Xmat, Y, lam, penalize_intercept=False)
    mse, r2 = metrics(Xmat, Y, theta)
    slope = theta[1]  
    rows.append({"lambda": lam, "slope": slope, "mse": mse, "r2": r2})

sim_results = pd.DataFrame(rows)
print(sim_results)

   lambda     slope       mse        r2
0       0  1.943135  3.701054  0.569934
1       1  1.941641  3.701057  0.569933
2      10  1.928291  3.701340  0.569901
3     100  1.804241  3.726114  0.567022
4    1000  1.097926  4.629031  0.462102
5   10000  0.223394  7.542853  0.123513
